# Transformers & HuggingFace: Tokenizers, Pretrained Models, Fine-tuning

Extends `06-deep-learning/05-attention-transformers`'s from-scratch NumPy self-attention and Keras Transformer-encoder work into the practical HuggingFace ecosystem: real subword tokenization, a real pretrained-model `pipeline()`, and a real small fine-tuning run compared against `07-nlp/04-deep-learning-nlp`'s from-scratch-trained Embedding+LSTM baseline on IMDB sentiment classification.

Models used are intentionally small (`distilbert-base-uncased`, 66M parameters) and every run here is CPU-only, small-data, few-epoch — a teaching demo, not a benchmark.

In [1]:
import time
import numpy as np
import torch
from torch.utils.data import DataLoader

torch.manual_seed(42)
np.random.seed(42)

print("torch:", torch.__version__)
import transformers, datasets, tokenizers
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("tokenizers:", tokenizers.__version__)

torch: 2.13.0+cpu


transformers: 5.15.1
datasets: 5.0.1
tokenizers: 0.22.2


## Part 1 — Subword tokenization: real splits on a real sentence

`notes.md`'s "Conceptual foundation" derives why word-level and character-level tokenization both fail, and why subword tokenization (WordPiece here, used by BERT/DistilBERT) is the practical middle ground. This cell loads DistilBERT's actual pretrained `WordPiece` tokenizer and applies it to a sentence chosen to contain both common whole words and a rare/compound word that must be split into subword pieces.

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sentence = "Transformers use subword tokenization, which handles unbelievably rare words gracefully."

tokens = tokenizer.tokenize(sentence)
encoded = tokenizer(sentence)

print("Sentence:", sentence)
print()
print("Subword tokens:", tokens)
print()
print("Token count:", len(tokens), "| Word count (naive whitespace split):", len(sentence.split()))
print()
print("input_ids:", encoded["input_ids"])
print("Decoded back:", tokenizer.decode(encoded["input_ids"]))

Sentence: Transformers use subword tokenization, which handles unbelievably rare words gracefully.

Subword tokens: ['transformers', 'use', 'sub', '##word', 'token', '##ization', ',', 'which', 'handles', 'un', '##bel', '##ie', '##va', '##bly', 'rare', 'words', 'gracefully', '.']

Token count: 18 | Word count (naive whitespace split): 10

input_ids: [101, 19081, 2224, 4942, 18351, 19204, 3989, 1010, 2029, 16024, 4895, 8671, 2666, 3567, 6321, 4678, 2616, 28266, 1012, 102]
Decoded back: [CLS] transformers use subword tokenization, which handles unbelievably rare words gracefully. [SEP]


**Reading the output:** `unbelievably` — not a common word — is split into `un`, `##bel`, `##ie`, `##va`, `##bly`; the `##` prefix marks "continues the previous token, no space before it." Every other, more common word (`transformers`, `subword`, `tokenization`, `rare`, `gracefully`) stays as one whole-word token. This is WordPiece's core behavior: common words stay atomic (short sequences, like word-level tokenization), rare/compound words decompose into known sub-pieces instead of becoming a single opaque `[UNK]` (no out-of-vocabulary problem, like character-level tokenization) — trading a few extra tokens for a rare word against never losing information to an unknown-word placeholder. See `notes.md`'s "Conceptual foundation" for the full OOV-vs-sequence-length tradeoff derivation.

## Part 2 — `pipeline()`: a real pretrained model, real output

`pipeline()` wraps three steps — tokenize, forward pass through a pretrained model, decode the model's output into a human-readable label — into one call. This cell runs `sentiment-analysis` with a small DistilBERT model already fine-tuned for that exact task (`distilbert-base-uncased-finetuned-sst-2-english`), on sentences the model has never seen, with the manual tokenizer/model steps shown side by side with the one-line `pipeline()` call to make the "what pipeline() does under the hood" mapping explicit.

In [3]:
from transformers import pipeline, AutoModelForSequenceClassification

sst2_model_name = "distilbert-base-uncased-finetuned-sst-2-english"

# --- manual: tokenize -> model forward pass -> decode, the three steps pipeline() automates ---
sst2_tokenizer = AutoTokenizer.from_pretrained(sst2_model_name)
sst2_model = AutoModelForSequenceClassification.from_pretrained(sst2_model_name)

test_sentences = [
    "This movie was surprisingly delightful and full of heart.",
    "Terrible plot, wooden acting, a complete waste of time.",
    "It was fine. Not great, not awful, just fine.",
]

manual_inputs = sst2_tokenizer(test_sentences, padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    manual_logits = sst2_model(**manual_inputs).logits
manual_probs = torch.softmax(manual_logits, dim=-1)
manual_preds = manual_probs.argmax(dim=-1)

print("--- manual tokenize -> forward -> decode ---")
for sent, pred, probs in zip(test_sentences, manual_preds, manual_probs):
    label = sst2_model.config.id2label[pred.item()]
    print(f"{label:>8}  ({probs[pred]:.4f})  {sent}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

--- manual tokenize -> forward -> decode ---
POSITIVE  (0.9999)  This movie was surprisingly delightful and full of heart.
NEGATIVE  (0.9998)  Terrible plot, wooden acting, a complete waste of time.
POSITIVE  (0.9987)  It was fine. Not great, not awful, just fine.


In [4]:
# --- the same three steps, via pipeline() ---
classifier = pipeline("sentiment-analysis", model=sst2_model_name)
pipeline_output = classifier(test_sentences)

print("--- pipeline(\"sentiment-analysis\") ---")
for sent, result in zip(test_sentences, pipeline_output):
    print(f"{result['label']:>8}  ({result['score']:.4f})  {sent}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

--- pipeline("sentiment-analysis") ---
POSITIVE  (0.9999)  This movie was surprisingly delightful and full of heart.
NEGATIVE  (0.9998)  Terrible plot, wooden acting, a complete waste of time.
POSITIVE  (0.9987)  It was fine. Not great, not awful, just fine.


The manual path and `pipeline()` agree exactly (same tokenizer, same model, same three steps) — `pipeline()` is not doing anything new mechanistically, it is packaging: `AutoTokenizer(...)` → `AutoModelForSequenceClassification(...)` forward pass → `softmax` + `argmax` + `id2label` lookup, in one call.

**From-scratch pointer:** none of the attention computation inside `sst2_model`'s forward pass is re-derived here — it is the exact scaled dot-product self-attention mechanism (`softmax(QK^T/√d_k)V`, multi-head, residual + layer norm) built from scratch in NumPy and then in Keras in [`06-deep-learning/05-attention-transformers`](../../06-deep-learning/05-attention-transformers/notes.md). What is new in this topic is everything *around* that mechanism: real WordPiece tokenization (Part 1, above) feeding it, and real pretrained weights (learned from a training run this repository did not do) filling every $W^Q, W^K, W^V, W^O$ matrix instead of the from-scratch topic's fixed-seed random or freshly-initialized-and-IMDB-trained ones.

## Part 3 — Fine-tuning: pretrained-then-fine-tuned vs. from-scratch-trained

**Baseline being compared against:** `07-nlp/04-deep-learning-nlp` trained an `Embedding(10000, 32) → LSTM(32) → Dense(1, sigmoid)` model **from randomly-initialized weights** on 20,000 IMDB training reviews for 8 epochs (early-stopped), reaching **86.94% test accuracy** on the full 25,000-review IMDB test set (see that topic's `notes.md`, "Experiment").

**Hypothesis (stated before running):** fine-tuning a pretrained DistilBERT — which already encodes general-purpose linguistic knowledge from large-scale pretraining, unlike the from-scratch LSTM's randomly-initialized embedding+recurrence — on a *much smaller* labeled subset than the from-scratch baseline used should still reach competitive sentiment-classification accuracy, and reach it with far less task-specific data and far fewer epochs, because fine-tuning only has to adapt existing knowledge rather than learn language structure and sentiment vocabulary from nothing.

**Setup:** `distilbert-base-uncased` (pretrained, no task head) + a fresh randomly-initialized 2-class classification head, fine-tuned with a plain PyTorch training loop (`AdamW`, no `Trainer`, to keep the actual gradient-update steps visible) on **400** IMDB training reviews (2.5% of the 20,000 the from-scratch baseline used to reach 86.94%), sequences truncated/padded to 96 WordPiece tokens, batch size 16, learning rate 3e-5, for 3 epochs — evaluated after every epoch on a held-out 200-review subset of the IMDB test split, to make the overfitting trend visible rather than reporting only a final number.

In [5]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")

N_TRAIN = 400
N_EVAL = 200
MAX_LEN = 96

train_raw = imdb["train"].shuffle(seed=42).select(range(N_TRAIN))
eval_raw = imdb["test"].shuffle(seed=42).select(range(N_EVAL))

ft_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_batch(batch):
    return ft_tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

train_ds = train_raw.map(tokenize_batch, batched=True).rename_column("label", "labels")
eval_ds = eval_raw.map(tokenize_batch, batched=True).rename_column("label", "labels")
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
eval_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Fine-tuning train set: {len(train_ds)} reviews ({len(train_ds)/20000:.1%} of the from-scratch baseline's 20,000)")
print(f"Held-out eval set: {len(eval_ds)} reviews")
print("Example tokenized length check:", len(train_ds[0]["input_ids"]), "tokens")

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Fine-tuning train set: 400 reviews (2.0% of the from-scratch baseline's 20,000)
Held-out eval set: 200 reviews
Example tokenized length check: 96 tokens


In [6]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
model.train()

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
eval_loader = DataLoader(eval_ds, batch_size=32)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

def evaluate(model, loader):
    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for batch in loader:
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            preds = logits.argmax(dim=-1)
            correct += (preds == batch["labels"]).sum().item()
            n += len(batch["labels"])
    model.train()
    return correct / n

EPOCHS = 3
history = []
t0 = time.time()

# accuracy of the pretrained head *before* any fine-tuning (randomly-initialized classifier on top of pretrained DistilBERT)
pre_ft_acc = evaluate(model, eval_loader)
print(f"Before fine-tuning (random classification head): eval accuracy = {pre_ft_acc:.4f}  [{time.time()-t0:.0f}s]")

for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    avg_train_loss = total_loss / len(train_loader)
    eval_acc = evaluate(model, eval_loader)
    history.append((epoch, avg_train_loss, eval_acc))
    print(f"epoch {epoch}: avg train loss = {avg_train_loss:.4f} | eval accuracy = {eval_acc:.4f}  [{time.time()-t0:.0f}s elapsed]")

print()
print(f"Total fine-tuning wall time: {time.time()-t0:.0f}s")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Before fine-tuning (random classification head): eval accuracy = 0.4600  [6s]


epoch 0: avg train loss = 0.6951 | eval accuracy = 0.5500  [110s elapsed]


epoch 1: avg train loss = 0.6524 | eval accuracy = 0.7050  [216s elapsed]


epoch 2: avg train loss = 0.4309 | eval accuracy = 0.7800  [324s elapsed]

Total fine-tuning wall time: 324s


## Experiment result and interpretation

**Real result (from the executed cell above):**

| Stage | Eval accuracy |
|---|---|
| Before fine-tuning (pretrained encoder + random classification head, no gradient updates yet) | 46.00% |
| After epoch 1 | 55.00% |
| After epoch 2 | 70.50% |
| After epoch 3 | **78.00%** |

Total fine-tuning wall time: 324s (~5.4 minutes) on CPU, for 3 epochs over 400 reviews.

**Interpretation:** the pre-fine-tuning accuracy (46.00%) is close to chance (50%) — as expected, since only the classification head is random at that point and it has seen zero gradient updates; DistilBERT's pretrained *encoder* representations are already there, but nothing has yet learned to read them for this specific task. Accuracy then rises every epoch (46.00% → 55.00% → 70.50% → 78.00%), with no decline in this 3-epoch run — unlike `07-nlp/04-deep-learning-nlp`'s from-scratch model, which had already started overfitting (validation accuracy declining) by epoch 2 out of 8. This is consistent with the hypothesis: 3 epochs over 400 labeled reviews (2% of the from-scratch baseline's 20,000) was enough to reach 78.00% accuracy, a competitive result reached with dramatically less labeled data and wall-clock training time than the 86.94%-accuracy from-scratch baseline needed (20,000 reviews, 8 epochs). It falls short of matching 86.94% outright — expected, given 50x less data and roughly a third of the epochs — but the *trajectory* (rising, not yet plateaued or overfitting at epoch 3) suggests a few more epochs or a larger fine-tuning subset would likely close more of that gap, which the from-scratch model could not do simply by adding epochs (it was already overfitting by epoch 2).

**Limitations:** one fine-tuning configuration (no hyperparameter search), a 400/200-review train/eval split rather than the full IMDB dataset (kept small deliberately, per `AGENTS.md`'s "no heavy/long-running training" constraint and this phase's few-minutes CPU runtime budget), the eval subset doubles as both per-epoch monitoring set and the final reported metric rather than using separate validation/test splits, and only 3 epochs were run — the accuracy trend was still rising at epoch 3, so 78.00% likely understates what a few more epochs (or more fine-tuning data) would reach; the comparison against `07-nlp/04-deep-learning-nlp`'s 86.94% should be read as an order-of-magnitude, label-efficiency comparison, not a controlled, matched-protocol benchmark.

## Summary

| | Data source | Epochs | Trainable params start from | Eval/test accuracy |
|---|---|---|---|---|
| `07-nlp/04-deep-learning-nlp` (from scratch) | 20,000 IMDB reviews | 8 (early-stopped) | random init | 86.94% (test) |
| This topic (pretrained + fine-tuned) | 400 IMDB reviews (2% as much) | 3 | DistilBERT pretrained weights + random classification head | 78.00% (eval), up from 46.00% pre-fine-tuning |

The from-scratch model needed 50x more labeled data and started with zero linguistic knowledge; the fine-tuned model starts with everything DistilBERT already learned from its (much larger, unlabeled-for-this-task) pretraining corpus, and only had to adapt that knowledge to "is this review positive or negative" — reaching 78.00% accuracy from 400 labeled examples and 5.4 minutes of CPU training, with its accuracy still rising (not yet overfitting) at the point training stopped. See `notes.md` for the full conceptual account of why pretrain-then-fine-tune is the default production pattern.